# Archetypes Officiels YGOPRODeck (TOK-7)

Récupère la liste officielle Konami des archetypes via `/api/v7/archetypes.php`
et construit un mapping entre les noms de tournoi (hybrides) et les noms officiels.

**Tables produites :**
- `archetypes_official` — 640 archetypes officiels Konami
- `archetype_mapping` — mapping tournament_archetype → primary/secondary official

**Résultat :** `cards.archetype` 100% aligné. `meta_scores` : 81/122 direct match,
41 noms hybrides de tournoi mappés vers leurs composantes officielles.

In [ ]:
import requests, sqlite3, pandas as pd

r = requests.get('https://db.ygoprodeck.com/api/v7/archetypes.php', timeout=15)
official = sorted([a['archetype_name'] for a in r.json()])
official_set = set(official)
print(f'{len(official)} archetypes officiels récupérés')

con = sqlite3.connect('../data/yugioh.db')
con.execute('DROP TABLE IF EXISTS archetypes_official')
con.execute('CREATE TABLE archetypes_official (archetype_name TEXT PRIMARY KEY)')
con.executemany('INSERT INTO archetypes_official VALUES (?)', [(a,) for a in official])
con.commit()
print('archetypes_official sauvegardée')

In [ ]:
ms_archetypes = pd.read_sql("SELECT DISTINCT archetype FROM meta_scores", con)['archetype'].tolist()

def find_primary_official(name, official_list):
    name_lower = name.lower()
    matches = [(a, len(a)) for a in official_list if a.lower() in name_lower]
    matches.sort(key=lambda x: -x[1])
    top = [m[0] for m in matches[:2]]
    return (top[0] if top else None), (top[1] if len(top) > 1 else None)

# Corrections manuelles pour les faux positifs de l'algo
MANUAL = {
    'Buster Blader Dragon Link': ('Buster Blader',  'Dragon Maid'),
    'Dragon Link':               ('Dragon Maid',    None),
    'Chain Burn':                ('Unchained',       None),
    'Earth Machine':             ('Machina',         None),
    'HEROs':                     ('HERO',            None),
    'Spellcasters':              ('Spellbook',       None),
    'Radiant Typhoon Tenpai':    ('Radiant Typhoon', 'Tenpai Dragon'),
    'Goblin Biker':              ('Goblin Rider',    None),
    'Dinos':                     ('Dinowrestler',    None),
    'Synchrons':                 ('Synchron',        'Synchro'),
    'Gem-Knights':               ('Gem-Knight',      None),
}

rows = []
for arch in ms_archetypes:
    if arch in official_set:
        rows.append({'tournament_archetype': arch, 'primary_official': arch,
                     'secondary_official': None, 'is_hybrid': 0})
    elif arch in MANUAL:
        p, s = MANUAL[arch]
        rows.append({'tournament_archetype': arch, 'primary_official': p,
                     'secondary_official': s, 'is_hybrid': 1})
    else:
        p, s = find_primary_official(arch, official)
        rows.append({'tournament_archetype': arch, 'primary_official': p or arch,
                     'secondary_official': s, 'is_hybrid': 1})

mapping = pd.DataFrame(rows)
con.execute('DROP TABLE IF EXISTS archetype_mapping')
con.execute('''CREATE TABLE archetype_mapping (
    tournament_archetype TEXT PRIMARY KEY,
    primary_official     TEXT,
    secondary_official   TEXT,
    is_hybrid            INTEGER
)''')
mapping.to_sql('archetype_mapping', con, if_exists='append', index=False)
con.commit()

print(f'{len(mapping)} entrées ({mapping["is_hybrid"].sum()} hybrides)')
print(f'Match direct : {(mapping["is_hybrid"]==0).sum()}/{len(mapping)}')
mapping[mapping['is_hybrid']==1][['tournament_archetype','primary_official','secondary_official']]